# **Training Pipeline (Quy trình huấn luyện)**

    Quy trình huấn luyện seq2seq hoàn chỉnh cho tóm tắt văn bản tiếng Việt.

## Luồng huấn luyện:
    1. Tải cấu hình
    2. Cài đặt seed 
    3. Tải tokenizer & mô hình
    4. Áp dụng LoRA (tùy chọn) 
    5. Tải & tiền xử lý dữ liệu
    6. Xây dựng Seq2SeqTrainer 
    7. Huấn luyện (Train) 
    8. Lưu mô hình tốt nhất
    9. Đánh giá (Evaluate) 
    10. Lưu các chỉ số (metrics)

## Hỗ trợ:
    - Fine-tuning toàn bộ hoặc huấn luyện hiệu quả tham số bằng LoRA
    - Huấn luyện đa GPU (Multi-GPU) qua HuggingFace Accelerate
    - Độ chính xác hỗn hợp (Mixed precision) (fp16/bf16/fp32 với tính năng tự động phát hiện)
    - Dừng sớm (Early stopping) dựa trên ROUGE-L trên tập đánh giá (validation)
    - Tiếp tục (Resume) từ checkpoint
    - Gradient checkpointing để tiết kiệm bộ nhớ

## Ví dụ:
    from src.trainer import train
    from src.config import load_config
    config = load_config("configs/vit5_base.yaml")
    train(config)  # Chạy toàn bộ quy trình huấn luyện

## 1. Clone repo & chuyển vào thư mục project

In [2]:
import os

REPO_URL = "https://github.com/dungcony/sumarization.git"
REPO_DIR = "/content/sumarization"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

Cloning into '/content/sumarization'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 43 (delta 5), reused 43 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 54.45 KiB | 3.63 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/sumarization


## 2. Cài đặt dependencies

In [3]:
%pip install -e .

Obtaining file:///content/sumarization
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 76, in resolve
    collected = self.factory.collect_root_requirements(root_reqs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/

## 3. Import các thư viện

In [ ]:
from __future__ import annotations

import argparse
import sys
from pathlib import Path

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.config import SummarizationConfig, config_to_dict
from src.data import load_and_preprocess
from src.evaluator import build_compute_metrics
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.utils import (
    detect_precision,
    format_duration,
    format_number,
    get_device_info,
    save_json,
    set_seed,
    setup_logger,
)